# Cálculo de KPIs en Python (después de la limpieza)

Este notebook replica las consultas del fichero `Calculo KPIs.sql` usando **pandas**, con los datos **ya limpios**.

**Requisito:** ejecuta antes el notebook `Limpieza de datos.ipynb` para generar `datos/tablas_limpias.pkl`.

## 1. Importar librerías

In [1]:
import pandas as pd

## 2. Cargar tablas limpias en DataFrames

Leemos las tablas desde `datos/tablas_limpias.pkl` (generado por `Limpieza de datos.ipynb`).

In [2]:
from DB import cargar_datos

datos = cargar_datos("datos/tablas_limpias.pkl")

print(f"Tablas cargadas desde fichero: {len(datos)}")
for nombre, df in datos.items():
    print(f"  - {nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

Tablas cargadas desde fichero: 8
  - cliente: 34 filas x 2 columnas
  - lote: 69567 filas x 2 columnas
  - marca: 35 filas x 2 columnas
  - producto: 70409 filas x 9 columnas
  - proveedor: 35 filas x 2 columnas
  - subtipo: 27 filas x 3 columnas
  - tipo: 15 filas x 2 columnas
  - venta: 70409 filas x 6 columnas


### 2.1 Crear variables para cada tabla

In [4]:
df_cliente = datos["cliente"]
df_lote = datos["lote"]
df_marca = datos["marca"]
df_producto = datos["producto"]
df_proveedor = datos["proveedor"]
df_subtipo = datos["subtipo"]
df_tipo = datos["tipo"]
df_venta = datos["venta"]

### 2.2 Preparar columnas útiles

In [7]:
# Copias para no modificar los DataFrames originales
venta = df_venta.copy()
producto = df_producto.copy()

# Fechas de venta
venta["tiempo_venta"] = pd.to_datetime(venta["tiempo_venta"])
venta["fecha_dia"] = venta["tiempo_venta"].dt.date
venta["mes"] = venta["tiempo_venta"].dt.month
venta["dia_mes"] = venta["tiempo_venta"].dt.day

# Valor de cada venta (precio por kilo * peso en gramos / 1000)
venta["valor_euros"] = venta["precio_venta"] * venta["peso"] / 1000

# Fechas de recogida
producto["tiempo_recogida"] = pd.to_datetime(producto["tiempo_recogida"])
producto["mes_recogida"] = producto["tiempo_recogida"].dt.month
producto["hora_recogida"] = producto["tiempo_recogida"].dt.hour

print("Datos preparados correctamente.")

Datos preparados correctamente.


---
## KPI 1) Media diaria de la cuantía de las distribuciones

**Interpretación 1:** número medio de distribuciones por día (solo ventas con fecha válida).

In [8]:
# Unir venta con producto (INNER JOIN)
venta_producto = venta.merge(producto, on="id_producto")

# Contar distribuciones por día
por_dia = (
    venta_producto.groupby("fecha_dia", as_index=False)
    .agg(nro_distribuciones=("id_lote", "count"))
)

# Solo días con fecha válida
por_dia = por_dia[por_dia["fecha_dia"].notna()]

media_distribuciones_diaria = round(por_dia["nro_distribuciones"].mean(), 2)
print(f"Media diaria de distribuciones: {media_distribuciones_diaria}")

Media diaria de distribuciones: 2133.61


**Interpretación 2:** si cuantía = valor económico de las ventas (precio por kilo).

In [9]:
valor_por_dia = (
    venta.groupby("fecha_dia", as_index=False)
    .agg(valor_venta=("valor_euros", "sum"))
)

valor_por_dia = valor_por_dia[valor_por_dia["fecha_dia"].notna()]

cuantia_media_diaria = round(valor_por_dia["valor_venta"].mean(), 2)
print(f"Cuantía media diaria (euros): {cuantia_media_diaria}")

Cuantía media diaria (euros): 2246.26


---
## KPI 2) Cuantía total de las distribuciones

In [10]:
total_distribuciones = por_dia["nro_distribuciones"].sum()
print(f"Total distribuciones (suma por días): {total_distribuciones}")

total_lotes_distintos = producto["id_lote"].nunique()
print(f"Total lotes distintos en producto: {total_lotes_distintos}")

Total distribuciones (suma por días): 70409
Total lotes distintos en producto: 69432


In [11]:
cuantia_total = round(valor_por_dia["valor_venta"].sum(), 2)
print(f"Cuantía total (euros): {cuantia_total}")

Cuantía total (euros): 74126.55


---
## KPI 3) Días del mes con más distribuciones (septiembre)

Top 5 días del mes con más distribuciones.

In [12]:
venta_sep = venta_producto[venta_producto["mes"] == 9]

dias_mas_distribuciones = (
    venta_sep.groupby("dia_mes", as_index=False)
    .agg(nro_distribuciones=("id_lote", "count"))
    .sort_values("nro_distribuciones", ascending=False)
    .head(5)
)

dias_mas_distribuciones.columns = ["fecha_dia", "nro_distribuciones"]
dias_mas_distribuciones

,fecha_dia,nro_distribuciones
12,13,2384
2,3,2383
17,18,2380
13,14,2377
21,22,2372


---
## KPI 4) Horas del día con más recogidas (septiembre)

Top 5 horas con más recogidas de alimentos.

In [13]:
producto_sep = producto[producto["mes_recogida"] == 9]

horas_mas_recogidas = (
    producto_sep.groupby("hora_recogida", as_index=False)
    .agg(nro_de_recogidas=("id_producto", "count"))
    .sort_values("nro_de_recogidas", ascending=False)
    .head(5)
)

horas_mas_recogidas

,hora_recogida,nro_de_recogidas
13,13,2995
21,21,2991
18,18,2989
19,19,2988
10,10,2986


---
## KPI 5) Top 5 clientes que más dinero han gastado (septiembre)

In [14]:
venta_cliente = venta[venta["mes"] == 9].merge(df_cliente, on="id_cliente")

top5_clientes_gasto = (
    venta_cliente.groupby("nombre", as_index=False)
    .agg(gasto_por_cliente=("valor_euros", "sum"))
    .sort_values("gasto_por_cliente", ascending=False)
    .head(5)
)

top5_clientes_gasto["gasto_por_cliente"] = top5_clientes_gasto["gasto_por_cliente"].round(2)
top5_clientes_gasto.columns = ["nombre_cliente", "gasto_por_cliente"]
top5_clientes_gasto

,nombre_cliente,gasto_por_cliente
27,SuperEconómico,2234.22
28,SuperMercado Ideal,2219.60
23,MegaCompra,2193.99
7,Cosecha Fresca,2187.14
9,Distribuciones del Sol,2185.68


---
## KPI 6) Top 5 clientes que menos dinero han gastado (septiembre)

In [15]:
top5_clientes_menos_gasto = (
    venta_cliente.groupby("nombre", as_index=False)
    .agg(gasto_por_cliente=("valor_euros", "sum"))
    .sort_values("gasto_por_cliente", ascending=True)
    .head(5)
)

top5_clientes_menos_gasto["gasto_por_cliente"] = top5_clientes_menos_gasto["gasto_por_cliente"].round(2)
top5_clientes_menos_gasto.columns = ["nombre_cliente", "gasto_por_cliente"]
top5_clientes_menos_gasto

,nombre_cliente,gasto_por_cliente
11,Distribuidora Gourmet,2030.71
2,Almacén Estrella,2037.88
3,Central de Abastos Central,2075.91
31,Supermercados del Valle,2076.45
12,Distribuidora Nacional,2080.92


---
## KPI 7) Top 10 proveedores que más dinero han recibido (septiembre)

In [16]:
venta_proveedor = (
    venta[venta["mes"] == 9]
    .merge(producto[["id_producto", "id_proveedor"]], on="id_producto")
    .merge(df_proveedor, on="id_proveedor")
)

top10_proveedores = (
    venta_proveedor.groupby("nombre", as_index=False)
    .agg(cuantia_ingreso_proveedor=("valor_euros", "sum"))
    .sort_values("cuantia_ingreso_proveedor", ascending=False)
    .head(10)
)

top10_proveedores["cuantia_ingreso_proveedor"] = top10_proveedores["cuantia_ingreso_proveedor"].round(2)
top10_proveedores.columns = ["nombre_proveedor", "cuantia_ingreso_proveedor"]
top10_proveedores

,nombre_proveedor,cuantia_ingreso_proveedor
13,EcoFungicidas Morales,2137.47
22,Maquinaria Agrícola Avanzada,2125.54
32,Tractores Eficientes Delgado,2116.98
20,Insecticidas Ecológicos Reyes,2116.53
3,AgroSoftware Soluciones,2112.16
0,Agricultura Inteligente TechCultivos,2109.30
7,Apícola EcoMiel,2108.88
5,Agronutrientes del Futuro,2106.80
2,AgroQuímicos Naturales BioCultivos,2096.21
12,Cultivos Hidropónicos Innovagro,2095.48


---
## KPI 8) Top 3 productos con mayor beneficio (septiembre)

Beneficio = precio de venta − coste inicial.

In [17]:
venta_tipo = (
    venta[venta["mes"] == 9]
    .merge(producto[["id_producto", "id_tipo", "coste_inicial"]], on="id_producto")
    .merge(df_tipo, on="id_tipo")
)

venta_tipo["margen"] = venta_tipo["precio_venta"] - venta_tipo["coste_inicial"]

top3_mayor_beneficio = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(margen_producto=("margen", "sum"))
    .sort_values("margen_producto", ascending=False)
    .head(3)
)

top3_mayor_beneficio.columns = ["tipo_fruta", "margen_producto"]
top3_mayor_beneficio

,tipo_fruta,margen_producto
3,Guava,28821.344723
0,Apple,16254.875117
4,Kiwi,12470.311929


---
## KPI 9) Top 3 productos con peor beneficio (septiembre)

In [18]:
top3_peor_beneficio = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(margen_producto=("margen", "sum"))
    .sort_values("margen_producto", ascending=True)
    .head(3)
)

top3_peor_beneficio.columns = ["tipo_fruta", "margen_producto"]
top3_peor_beneficio

,tipo_fruta,margen_producto
2,Carambola,3034.166793
14,muskmelon,3043.926008
9,Persimmon,3075.914354


---
## KPI 10) Precio de venta medio de cada fruta (septiembre)

In [19]:
precio_medio_fruta = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(precio_medio=("precio_venta", "mean"))
    .sort_values("precio_medio", ascending=True)
)

precio_medio_fruta.columns = ["producto", "precio_medio"]
precio_medio_fruta

,producto,precio_medio
13,Tomatoes,3.450088
2,Carambola,3.480256
11,Plum,3.485459
0,Apple,3.486805
14,muskmelon,3.487770
1,Banana,3.494400
5,Mango,3.498009
3,Guava,3.501339
9,Persimmon,3.501774
12,Pomegranate,3.507570


---
## KPI 11) Fruta dañada por tipo (septiembre)

Ventas sin precio = fruta que no se pudo vender (dañada en la distribución).

In [21]:
venta_danada = venta_tipo[(venta_tipo["mes"] == 9) & (venta_tipo["precio_venta"].isna())]

fruta_danada_por_tipo = (
    venta_danada.groupby("nombre", as_index=False)
    .agg(nro_ventas_no_exitosas=("id_venta", "count"))
    .sort_values("nro_ventas_no_exitosas", ascending=False)
)

fruta_danada_por_tipo.columns = ["producto", "nro_ventas_no_exitosas"]
fruta_danada_por_tipo

,producto,nro_ventas_no_exitosas


---
## KPI 12) Pérdida total de la fruta dañada (septiembre)

In [22]:
venta_danada["coste"] = venta_danada["coste_inicial"] * venta_danada["peso"] / 1000

coste_ventas_no_exitosas = round(venta_danada["coste"].sum(), 2)
print(f"Pérdida total fruta dañada (euros): {coste_ventas_no_exitosas}")

Pérdida total fruta dañada (euros): 0.0


---
## KPI 13) Cuantía por tipo de fruta de los 5 clientes que más gastan

Primero identificamos los 5 clientes con más gasto (en todo el histórico con fecha válida, como en el SQL).
Después calculamos cuánto gastaron en cada fruta en septiembre.

In [23]:
# Top 5 clientes por gasto total (SQL: MONTH IS NOT NULL)
venta_con_fecha = venta[venta["mes"].notna()].copy()

top5_ids = (
    venta_con_fecha.groupby("id_cliente")["valor_euros"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

# Renombramos para evitar confusión con el nombre de la fruta
cliente_nombres = df_cliente.rename(columns={"nombre": "cliente"})

# Gasto por cliente y tipo de fruta en septiembre
venta_top5 = venta_tipo[
    (venta_tipo["mes"] == 9) & (venta_tipo["id_cliente"].isin(top5_ids))
].merge(cliente_nombres, on="id_cliente")

cuantia_top5_clientes = (
    venta_top5.groupby(["cliente", "nombre"], as_index=False)
    .agg(cuantia_gastada=("valor_euros", "sum"))
    .sort_values(["cliente", "cuantia_gastada"], ascending=[True, False])
)

cuantia_top5_clientes["cuantia_gastada"] = cuantia_top5_clientes["cuantia_gastada"].round(2)
cuantia_top5_clientes.columns = ["cliente", "tipo_fruta", "cuantia_gastada"]
cuantia_top5_clientes

,cliente,tipo_fruta,cuantia_gastada
3,Cosecha Fresca,Guava,649.65
0,Cosecha Fresca,Apple,353.09
4,Cosecha Fresca,Kiwi,243.65
5,Cosecha Fresca,Mango,140.57
8,Cosecha Fresca,Pear,81.98
...,...,...,...
70,SuperMercado Ideal,Pitaya,69.39
73,SuperMercado Ideal,Tomatoes,69.22
72,SuperMercado Ideal,Pomegranate,64.74
69,SuperMercado Ideal,Persimmon,64.57


---
## KPI 14) Porcentaje de beneficio por producto (septiembre)

In [24]:
venta_tipo["porcentaje_beneficio"] = (
    (venta_tipo["precio_venta"] - venta_tipo["coste_inicial"]) * 100 / venta_tipo["coste_inicial"]
)

beneficio_por_tipo = (
    venta_tipo.groupby("nombre", as_index=False)
    .agg(porcentaje_beneficio=("porcentaje_beneficio", "mean"))
    .sort_values("porcentaje_beneficio", ascending=False)
)

beneficio_por_tipo["porcentaje_beneficio"] = (
    beneficio_por_tipo["porcentaje_beneficio"].round(2).astype(str) + "%"
)
beneficio_por_tipo.columns = ["tipo_fruta", "porcentaje_beneficio"]
beneficio_por_tipo

,tipo_fruta,porcentaje_beneficio
2,Carambola,85.41%
12,Pomegranate,82.82%
10,Pitaya,82.45%
9,Persimmon,82.45%
8,Pear,81.93%
4,Kiwi,81.44%
1,Banana,81.27%
14,muskmelon,81.23%
6,Orange,81.22%
7,Peach,81.07%
